# CLAMPfull pseudobulk disentanglement proof

Checks that CLAMPfull's latent variables (LVs) disentangle cell types: each cell
type correlates with one distinct LV (from `00_benchmark`), and that LV's
top-loading genes independently recover markers of the same cell type using a
combined CellMarker, Allen Brain Atlas, and dataset-matched Azimuth reference.

💡 **Environment:** `clamp-analyses`

## Libraries

In [ ]:
library(here)
library(yaml)
library(dplyr)
library(tidyr)
library(ggplot2)
library(patchwork)
library(stringr)

source(here("scripts", "pseudobulk", "common.R"))


## Settings

In [ ]:
DATASETS     <- snakemake@params[["datasets"]]
MOD_ROOT     <- here(snakemake@params[["mod_root"]])
OUT_DIR      <- here(snakemake@params[["out_dir"]])
TOP_GENE_PCT          <- as.numeric(snakemake@params[["top_gene_fraction"]])
RETRY_GENE_PCT        <- as.numeric(snakemake@params[["retry_gene_fraction"]])
FDR_THRESH            <- as.numeric(snakemake@params[["fdr_threshold"]])
TOP_N_PATHWAYS        <- as.integer(snakemake@params[["top_terms"]])

dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)
set.seed(42)

pseudobulk_dir <- function(ds) {
  cfg <- snakemake@config$datasets[[ds]]
  if (isTRUE(cfg$build_pseudobulk)) {
    file.path(MOD_ROOT, ds, 'pseudobulk')
  } else {
    here(cfg$pseudobulk_dir)
  }
}

truth_v0_path <- function(ds) {
  path <- file.path(pseudobulk_dir(ds), "truthFrac_v0.csv")
  if (!file.exists(path))
    stop("truthFrac_v0.csv not found for ", ds, " at ", path)
  path
}

ct_labels_df <- read.csv(here("data", "pseudobulk", "cell_type_labels.csv"),
                         stringsAsFactors = FALSE)
CT_LABELS    <- setNames(ct_labels_df$label, ct_labels_df$cell_type)
ct_label     <- function(x) ifelse(x %in% names(CT_LABELS), CT_LABELS[x], x)

## Each cell type correlates with a distinct LV

In [ ]:
ASSIGNMENTS_PATH <- snakemake@input[["assignments"]]

top_lvs_df <- read.csv(ASSIGNMENTS_PATH, stringsAsFactors = FALSE)

top_lvs_df <- top_lvs_df[top_lvs_df$dataset %in% DATASETS, ]
rownames(top_lvs_df) <- NULL

cat("CLAMPfull LV assignments loaded:", nrow(top_lvs_df), "rows\n")
cat("Datasets:\n")
print(table(top_lvs_df$dataset))
cat("\nSample (first 10 rows):\n")
print(head(top_lvs_df, 10))

## Different cell types are captured by different LVs

In [ ]:
CORR_FULL_PATH <- snakemake@input[["corr"]]
lv_corr_full   <- read.csv(CORR_FULL_PATH, stringsAsFactors = FALSE)
lv_corr_full   <- lv_corr_full[lv_corr_full$dataset %in% DATASETS, ]

cat("Full LV x cell-type correlation matrix loaded:", nrow(lv_corr_full), "rows\n")

heatmap_panels <- lapply(DATASETS, function(ds) {
    d <- lv_corr_full[lv_corr_full$dataset == ds, ]
    if (nrow(d) == 0) return(NULL)

    d <- d[!is.na(d$cell_type) & d$cell_type != "NA", ]
    assigned <- top_lvs_df[top_lvs_df$dataset == ds, c("cell_type", "LV")]
    assigned <- assigned[!is.na(assigned$cell_type) & assigned$cell_type != "NA", ]
    if (nrow(d) == 0 || nrow(assigned) == 0) return(NULL)
    d <- d[d$LV %in% assigned$LV & d$cell_type %in% assigned$cell_type, ]
    d$cell_type_label <- ct_label(d$cell_type)

    ord      <- order(assigned$cell_type)
    lv_order <- unique(assigned$LV[ord])
    ct_order <- unique(ct_label(assigned$cell_type[ord]))
    d$LV              <- factor(d$LV, levels = lv_order)
    d$cell_type_label <- factor(d$cell_type_label, levels = ct_order)

    p <- ggplot(d, aes(x = cell_type_label, y = LV, fill = cor)) +
        geom_tile(color = "white", linewidth = 0.4) +
        geom_text(aes(label = sprintf("%.2f", cor)), size = 2.5) +
        scale_fill_gradientn(colours = c("#b2182b", "#fdf7f7", "white", "#f7fbf7", "#007a33"),
                             values = scales::rescale(c(-1, -0.5, 0, 0.5, 1)),
                             limits = c(-1, 1), name = "r") +
        coord_fixed() +
        theme_bw(base_size = 9) +
        theme(axis.text.x = element_text(angle = 45, hjust = 1),
              plot.title  = element_text(face = "bold", size = 10, hjust = 0.5)) +
        labs(x = NULL, y = NULL, title = ds)

    list(dataset = ds, plot = p, n_cell_types = length(ct_order), n_lvs = length(lv_order))
})
heatmap_panels <- Filter(Negate(is.null), heatmap_panels)

for (panel in heatmap_panels) {
    p <- panel$plot + theme(legend.position = "right")
    print(p)
}

In [ ]:
combined_heatmap <- patchwork::wrap_plots(
    lapply(heatmap_panels, `[[`, "plot"),
    ncol = 3, guides = "collect"
) & theme(legend.position = "right")
combined_heatmap

## Assignment quality, correlation, specificity, and one-vs-rest AUROC

In [ ]:
auroc_rows <- lapply(unique(top_lvs_df$dataset), function(ds) {
    b_path     <- file.path(MOD_ROOT, ds, "models", "CLAMPfull", "B.csv")
    truth_path <- truth_v0_path(ds)
    if (!file.exists(b_path)) return(NULL)

    B     <- t(as.matrix(read.csv(b_path, row.names = 1, check.names = FALSE)))
    truth <- filter_analysis_cell_types(
      read.csv(truth_path, row.names = 1, check.names = FALSE),
      ds, snakemake@config$cell_type_analysis$excluded_targets
    )
    sample_ct <- setNames(colnames(truth)[apply(truth, 1, which.max)], rownames(truth))
    shared    <- intersect(rownames(B), names(sample_ct))

    ds_rows <- top_lvs_df[top_lvs_df$dataset == ds, ]
    do.call(rbind, lapply(seq_len(nrow(ds_rows)), function(i) {
        ct <- ds_rows$cell_type[i]; lv <- ds_rows$LV[i]
        if (!lv %in% colnames(B)) return(NULL)
        in_idx <- sample_ct[shared] == ct
        x_in   <- B[shared, lv][in_idx]
        x_out  <- B[shared, lv][!in_idx]
        if (length(x_in) < 3 || length(x_out) < 3) return(NULL)
        wt <- suppressWarnings(wilcox.test(x_in, x_out))
        data.frame(dataset = ds, cell_type = ct, LV = lv,
                   auroc = as.numeric(wt$statistic) / (length(x_in) * length(x_out)),
                   stringsAsFactors = FALSE)
    }))
})
auroc_df <- do.call(rbind, auroc_rows)

# Tau specificity (Yanai et al. 2005)
specificity_rows <- lapply(seq_len(nrow(top_lvs_df)), function(i) {
    ds <- top_lvs_df$dataset[i]; lv <- top_lvs_df$LV[i]
    x  <- lv_corr_full$cor[lv_corr_full$dataset == ds & lv_corr_full$LV == lv]
    x[x < 0] <- 0
    n <- length(x)
    if (n < 2 || max(x) <= 0) return(data.frame(tau = NA_real_))
    xhat <- x / max(x)
    data.frame(tau = sum(1 - xhat) / (n - 1))
})
specificity_df <- do.call(rbind, specificity_rows)

summarize_metric <- function(x) data.frame(mean = mean(x, na.rm = TRUE), sd = sd(x, na.rm = TRUE))

assignment_quality <- dplyr::bind_rows(
    cbind(metric = "mean_cor_matched", summarize_metric(top_lvs_df$cor)),
    cbind(metric = "mean_tau",         summarize_metric(specificity_df$tau)),
    cbind(metric = "mean_auroc",       summarize_metric(auroc_df$auroc))
)
assignment_quality$label <- sprintf("%.3f ± %.3f", assignment_quality$mean, assignment_quality$sd)
assignment_quality

specificity_df$dataset <- top_lvs_df$dataset

assignment_quality_by_dataset <- dplyr::bind_rows(lapply(unique(top_lvs_df$dataset), function(ds) {
    dplyr::bind_rows(
        cbind(dataset = ds, metric = "mean_cor_matched",
              summarize_metric(top_lvs_df$cor[top_lvs_df$dataset == ds])),
        cbind(dataset = ds, metric = "mean_tau",
              summarize_metric(specificity_df$tau[specificity_df$dataset == ds])),
        cbind(dataset = ds, metric = "mean_auroc",
              summarize_metric(auroc_df$auroc[auroc_df$dataset == ds]))
    )
}))
assignment_quality_by_dataset$label <- sprintf("%.3f ± %.3f", assignment_quality_by_dataset$mean, assignment_quality_by_dataset$sd)
assignment_quality_by_dataset

## Module-level marker recovery, top-loading genes recover independent cell-type markers

In [ ]:
CELLMARKER_PATH <- snakemake@input[["cell_marker_file"]]
ALLEN_PATH      <- snakemake@input[["allen_brain_gmt_file"]]
AZIMUTH_PATH    <- snakemake@input[["azimuth_file"]]

load_cellmarker_sets <- function(path) {
    df <- readxl::read_excel(path, sheet = "human")
    df <- df[df$species == "Human" & !is.na(df$Symbol) & nzchar(df$Symbol), ]
    lapply(split(df$Symbol, df$cell_name), unique)
}

read_gmt_sets <- function(path) {
    sp <- strsplit(readLines(path, warn = FALSE), "\t", fixed = TRUE)
    sp <- sp[lengths(sp) >= 3]
    setNames(lapply(sp, function(x) unique(x[-c(1, 2)])),
             vapply(sp, `[`, character(1), 1))
}

cellmarker_t2g <- load_cellmarker_sets(CELLMARKER_PATH)
allen_raw <- read_gmt_sets(ALLEN_PATH)
allen_t2g <- allen_raw[grepl("^Human ", names(allen_raw)) &
                        grepl(" up$", names(allen_raw))]
BRAIN_DS <- unlist(snakemake@config$references$brain_datasets)

normalize_marker_text <- function(x) {
    stringr::str_squish(gsub("[^a-z0-9]+", " ", tolower(x)))
}

CELL_TYPE_ALIASES <- list(
    Ast = c("astrocyte"), Exc = c("excitatory neuron", "glutamatergic neuron"),
    Inh = c("inhibitory neuron", "gabaergic neuron"),
    Mic = c("microglia", "microglial"), Oli = c("oligodendrocyte"),
    Opc = c("oligodendrocyte precursor", "oligodendrocyte progenitor", "opc"),
    B_cell = c("b cell", "b lymphocyte"), NK = c("natural killer", "nk cell"),
    CD4_T = c("cd4 t cell", "cd4 positive t cell"),
    CD8_T = c("cd8 t cell", "cd8 positive t cell"),
    CD14_Mono = c("cd14 monocyte", "classical monocyte", "monocyte"),
    CD16_Mono = c("cd16 monocyte", "non classical monocyte", "monocyte"),
    DC = c("dendritic cell"), gd_T = c("gamma delta t cell", "gd t cell"),
    Plasma_B = c("plasma cell", "plasmablast"),
    Myeloid = c("myeloid", "monocyte", "macrophage"),
    T_cell = c("t cell", "t lymphocyte"),
    Macrophage = c("macrophage", "myeloid"),
    Endothelial = c("endothelial cell"), Pericyte = c("pericyte"),
    VentricularCM = c("ventricular cardiomyocyte", "cardiomyocyte"),
    AtrialCM = c("atrial cardiomyocyte", "cardiomyocyte"),
    Fibroblast = c("fibroblast"), Adipocyte = c("adipocyte"),
    Endocardial = c("endocardial cell", "endothelial cell"),
    Epicardial = c("epicardial cell", "mesothelial cell"),
    LymphaticEndothelial = c("lymphatic endothelial"),
    Lymphocyte = c("lymphocyte", "t cell", "b cell", "natural killer"),
    Mast = c("mast cell"), Neuronal = c("neuron", "neuronal"),
    VSMC = c("vascular smooth muscle", "smooth muscle cell"),
    `Blood vessels` = c("endothelial cell", "vascular endothelial"),
    Lymphoid = c("lymphocyte", "t cell", "b cell", "natural killer"),
    `Alveolar epithelium` = c("alveolar epithelial", "type i pneumocyte", "type ii pneumocyte"),
    `Airway epithelium` = c("ciliated cell", "club cell", "goblet cell", "basal cell"),
    `Fibroblast lineage` = c("fibroblast", "myofibroblast"),
    `Lymphatic EC` = c("lymphatic endothelial"),
    `Submucosal Gland` = c("submucosal gland", "serous cell", "mucous cell")
)

cellmarker_terms_for <- function(cell_type) {
    aliases <- CELL_TYPE_ALIASES[[cell_type]]
    if (is.null(aliases)) aliases <- ct_label(cell_type)
    aliases <- normalize_marker_text(aliases)
    terms_norm <- normalize_marker_text(names(cellmarker_t2g))
    names(cellmarker_t2g)[vapply(terms_norm, function(term)
        any(vapply(aliases, function(alias)
            grepl(paste0(" ", alias, " "), paste0(" ", term, " "),
                  fixed = TRUE), logical(1))), logical(1))]
}

ALLEN_ALIAS_MAP <- list(
    Ast = "Astro", Exc = "Exc", Inh = "Inh", Mic = "Micro",
    Oli = "Oligo", Opc = "OPC"
)

allen_terms_for <- function(dataset, cell_type) {
    if (!dataset %in% BRAIN_DS) return(character(0))
    prefixes <- ALLEN_ALIAS_MAP[[cell_type]]
    if (is.null(prefixes)) return(character(0))
    pattern <- paste0("^Human (", paste(prefixes, collapse = "|"), ") ")
    names(allen_t2g)[grepl(pattern, names(allen_t2g))]
}

read_azimuth_sets <- function(path) {
    sp <- strsplit(readLines(path, warn = FALSE), "\t", fixed = TRUE)
    sp <- sp[lengths(sp) >= 3]
    setNames(lapply(sp, function(x) unique(x[-c(1, 2)])),
             vapply(sp, `[`, character(1), 1))
}
azimuth_t2g <- read_azimuth_sets(AZIMUTH_PATH)

AZIMUTH_TERM_MAP <- list(
    Brain_Mathys2023 = list(
        Oli = "Motor Cortex-subclass-oligodendrocyte",
        Exc = "Motor Cortex-class-Glutamatergic Neuron",
        Ast = "Motor Cortex-subclass-Astrocyte",
        Inh = "Motor Cortex-class-GABAergic Neuron",
        Mic = "Motor Cortex-subclass-microglia / Perivascular Macrophage",
        Opc = "Motor Cortex-subclass-oligodendrocyte Precursor Cell"
    ),
    Brain_Xiong2023 = list(
        Oli = "Motor Cortex-subclass-oligodendrocyte",
        Exc = "Motor Cortex-class-Glutamatergic Neuron",
        Ast = "Motor Cortex-subclass-Astrocyte",
        Inh = "Motor Cortex-class-GABAergic Neuron",
        Mic = "Motor Cortex-subclass-microglia / Perivascular Macrophage",
        Opc = "Motor Cortex-subclass-oligodendrocyte Precursor Cell"
    ),
    Heart_Datar2026 = list(
        Macrophage = "Heart-L2-Macrophage",
        Endothelial = "Heart-L2-Endothelial",
        Pericyte = "Heart-L2-Pericyte",
        VentricularCM = "Heart-L2-Ventricular Cardiomyocyte",
        Fibroblast = "Heart-L2-Fibroblast",
        Adipocyte = "Heart-L2-Adipocyte",
        AtrialCM = "Heart-L2-Atrial Cardiomyocyte",
        Endocardial = "Heart-L2-Endocardial",
        Epicardial = "Heart-L2-Mesothelial",
        LymphaticEndothelial = "Heart-L2-Lymphatic Endothelial",
        Lymphocyte = c("Heart-L2-B", "Heart-L2-ILC", "Heart-L2-NK", "Heart-L2-T"),
        Mast = "Heart-L2-Mast",
        Neuronal = "Heart-L2-Neuronal",
        VSMC = "Heart-L2-Smooth Muscle"
    ),
    PBMC_1k1k = list(
        B_cell = "PBMC-L1-B Cell",
        NK = "PBMC-L1-natural Killer Cell",
        CD4_T = "PBMC-L1-CD4+ T Cell",
        CD8_T = "PBMC-L1-CD8+ T Cell",
        CD14_Mono = "PBMC-L2-CD14+ Monocyte",
        CD16_Mono = "PBMC-L2-CD16+ Monocyte",
        DC = "PBMC-L1-dendritic Cell",
        gd_T = "PBMC-L2-gamma-delta T",
        Plasma_B = "PBMC-L2-Plasmablast"
    ),
    PBMC_Perez2022 = list(
        Myeloid = c("PBMC-L1-Monocyte", "PBMC-L1-dendritic Cell"),
        T_cell = c("PBMC-L1-CD4+ T Cell", "PBMC-L1-CD8+ T Cell",
                   "PBMC-L1-other T Cell"),
        B_cell = "PBMC-L1-B Cell",
        NK = "PBMC-L1-natural Killer Cell"
    ),
    Lung_Sikkema2023 = list(
        "Blood vessels" = "Lung V2 (HLCA)-ann Level 2-Blood Vessels",
        Lymphoid = "Lung V2 (HLCA)-ann Level 2-Lymphoid",
        "Alveolar epithelium" = "Lung V2 (HLCA)-ann Level 2-Alveolar Epithelium",
        "Airway epithelium" = "Lung V2 (HLCA)-ann Level 2-Airway Epithelium",
        Myeloid = "Lung V2 (HLCA)-ann Level 2-Myeloid",
        "Fibroblast lineage" = "Lung V2 (HLCA)-ann Level 2-Fibroblast Lineage",
        "Lymphatic EC" = "Lung V2 (HLCA)-ann Level 2-Lymphatic EC",
        "Submucosal Gland" = "Lung V2 (HLCA)-ann Level 2-Submucosal Gland"
    )
)

mapped_terms <- unique(unlist(AZIMUTH_TERM_MAP, recursive = TRUE, use.names = FALSE))
missing_terms <- setdiff(mapped_terms, names(azimuth_t2g))
if (length(missing_terms))
    stop("Azimuth mapping contains missing terms: ", paste(missing_terms, collapse = "; "))

azimuth_terms_for <- function(dataset, cell_type) {
    ds_map <- AZIMUTH_TERM_MAP[[dataset]]
    if (is.null(ds_map) || is.null(ds_map[[cell_type]]))
        stop("No Azimuth marker mapping for ", dataset, " / ", cell_type)
    ds_map[[cell_type]]
}

azimuth_dataset_terms <- function(dataset, cell_types = names(AZIMUTH_TERM_MAP[[dataset]])) {
    unique(unlist(AZIMUTH_TERM_MAP[[dataset]][cell_types], use.names = FALSE))
}

prefix_marker_sets <- function(sets, source) {
    stats::setNames(sets, paste0(source, "::", names(sets)))
}

prefix_marker_terms <- function(terms, source) {
    if (length(terms) == 0) return(character(0))
    paste0(source, "::", terms)
}

reference_sets_for <- function(dataset, cell_types = names(AZIMUTH_TERM_MAP[[dataset]])) {
    az_terms <- azimuth_dataset_terms(dataset, cell_types)
    out <- c(prefix_marker_sets(cellmarker_t2g, "CellMarker"),
             prefix_marker_sets(azimuth_t2g[az_terms], "Azimuth"))
    if (dataset %in% BRAIN_DS)
        out <- c(out, prefix_marker_sets(allen_t2g, "Allen"))
    out
}

target_reference_terms <- function(dataset, cell_type) {
    unique(c(prefix_marker_terms(cellmarker_terms_for(cell_type), "CellMarker"),
             prefix_marker_terms(azimuth_terms_for(dataset, cell_type), "Azimuth"),
             prefix_marker_terms(allen_terms_for(dataset, cell_type), "Allen")))
}

build_marker_sets <- function(dataset, cell_types, gene_universe, min_size = 5,
                              promiscuity_max_frac = 0.5) {
    references <- reference_sets_for(dataset, cell_types)
    sets <- lapply(cell_types, function(ct) {
        terms <- intersect(target_reference_terms(dataset, ct), names(references))
        intersect(unique(unlist(references[terms], use.names = FALSE)), gene_universe)
    })
    names(sets) <- cell_types
    too_small <- names(sets)[lengths(sets) < min_size]
    if (length(too_small))
        stop("Combined marker sets below min_size for ", dataset, ": ",
             paste(too_small, collapse = ", "))

    if (length(sets) > 1) {
        gene_counts <- table(unlist(sets))
        promiscuous <- names(gene_counts)[gene_counts > promiscuity_max_frac * length(sets)]
        sets <- lapply(sets, function(g) setdiff(g, promiscuous))
        too_small <- names(sets)[lengths(sets) < min_size]
        if (length(too_small))
            stop("Combined marker sets below min_size after promiscuity filter for ",
                 dataset, ": ", paste(too_small, collapse = ", "))
    }
    sets
}

read_B_matrix <- function(path) {
    df  <- read.csv(path, row.names = 1, check.names = FALSE)
    mat <- as.matrix(df)
    storage.mode(mat) <- "numeric"
    t(mat)   # samples x LVs
}

cat("CellMarker sets:", length(cellmarker_t2g),
    "| Allen human-up sets:", length(allen_t2g),
    "| Azimuth sets:", length(azimuth_t2g),
    "| mapped Azimuth terms:", length(mapped_terms), "\n")

In [ ]:
run_module_pipeline <- function(ds, method = "CLAMPfull") {
    z_path     <- file.path(MOD_ROOT, ds, "models", method, "Z.csv")
    truth_path <- truth_v0_path(ds)
    if (!file.exists(z_path)) return(NULL)

    Z         <- read.csv(z_path, row.names = 1, check.names = FALSE)
    truth <- filter_analysis_cell_types(
      read.csv(truth_path, row.names = 1, check.names = FALSE),
      ds, snakemake@config$cell_type_analysis$excluded_targets
    )
    all_cts   <- colnames(truth)

    universe    <- rownames(Z)
    marker_sets <- build_marker_sets(ds, all_cts, universe)

    assigned <- top_lvs_df[top_lvs_df$dataset == ds & top_lvs_df$cell_type %in% names(marker_sets), ]
    if (nrow(assigned) != length(marker_sets)) stop("Incomplete correlation assignment for ", ds)
    if (anyDuplicated(assigned$cell_type) || anyDuplicated(assigned$LV)) stop("Non-unique correlation assignment for ", ds)
    sel <- data.frame(
        cell_type = assigned$cell_type, LV = assigned$LV,
        n_in = NA_real_, n_out = NA_real_, mean_in = NA_real_, mean_out = NA_real_,
        effect = assigned$cor, pvalue = NA_real_, qvalue = NA_real_,
        is_significant = NA, weak_evidence = NA,
        selection_method = "correlation_assignment", stringsAsFactors = FALSE
    )
    ovr <- data.frame(cell_type = character(), LV = character(), n_in = numeric(),
                      n_out = numeric(), mean_in = numeric(), mean_out = numeric(),
                      effect = numeric(), pvalue = numeric(), qvalue = numeric())

    list(dataset = ds, method = method, ovr = ovr, selected = sel,
         marker_sets = marker_sets, marker_source = "CellMarker + Allen + Azimuth",
         universe = universe)
}

module_results <- lapply(DATASETS, run_module_pipeline)
names(module_results) <- DATASETS
module_results <- Filter(Negate(is.null), module_results)

ovr_all_df <- data.frame(dataset = character(), cell_type = character(), LV = character(),
                         n_in = numeric(), n_out = numeric(), mean_in = numeric(),
                         mean_out = numeric(), effect = numeric(), pvalue = numeric(),
                         qvalue = numeric())
selected_all_df <- dplyr::bind_rows(lapply(names(module_results), function(ds)
    cbind(dataset = ds, module_results[[ds]]$selected)))

cat("Selected LVs (one per cell type per dataset, all cell types covered):", nrow(selected_all_df), "\n")
print(table(selected_all_df$selection_method))
print(selected_all_df[, c("dataset", "cell_type", "LV", "effect", "selection_method", "is_significant")])

In [ ]:
coverage_df <- selected_all_df[, c("dataset", "cell_type", "selection_method")]
stopifnot(nrow(selected_all_df) == 47L)
stopifnot(!anyDuplicated(paste(selected_all_df$dataset, selected_all_df$cell_type)))
stopifnot(!any(vapply(split(selected_all_df$LV, selected_all_df$dataset), anyDuplicated, integer(1)) > 0L))
print(table(coverage_df$selection_method))

In [ ]:
# ORA per selected LV against the combined CellMarker, Allen, and
# dataset-matched Azimuth marker reference. Allen is included for brain only.
# The top 1% of gene loadings is tested first; an LV with no significant hit is
# retried once at 3%.
suppressPackageStartupMessages(library(clusterProfiler))

get_top_genes <- function(Z, lv, top_pct) {
    if (!lv %in% colnames(Z)) return(character(0))
    loadings <- Z[[lv]]; names(loadings) <- rownames(Z)
    n_genes  <- max(1L, ceiling(length(loadings) * top_pct))
    ord      <- order(loadings, decreasing = TRUE)
    names(loadings)[ord][seq_len(n_genes)]
}

ORA_TERM_SEP          <- " || "  # CellMarker term names can contain commas

run_enricher_once <- function(Z, LV, pct, term2gene_full, universe) {
    top_genes <- get_top_genes(Z, LV, pct)
    tryCatch(
        clusterProfiler::enricher(gene = top_genes, universe = universe,
                                  TERM2GENE = term2gene_full, pvalueCutoff = 1,
                                  qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf),
        error = function(e) NULL)
}

significant_ora <- function(res) {
    if (is.null(res) || nrow(res@result) == 0) return(data.frame())
    res@result[res@result$p.adjust < FDR_THRESH, , drop = FALSE]
}

run_lv_ora_topn <- function(ds, LV, term2gene_full, universe) {
    z_path <- file.path(MOD_ROOT, ds, "models", "CLAMPfull", "Z.csv")
    Z      <- read.csv(z_path, row.names = 1, check.names = FALSE)

    res <- run_enricher_once(Z, LV, TOP_GENE_PCT, term2gene_full, universe)
    sig <- significant_ora(res)
    pct_used <- TOP_GENE_PCT
    if (nrow(sig) == 0) {
        res <- run_enricher_once(Z, LV, RETRY_GENE_PCT,
                                 term2gene_full, universe)
        sig <- significant_ora(res)
        pct_used <- RETRY_GENE_PCT
    }
    if (nrow(sig) == 0)
        return(data.frame(top_pathways = NA_character_, top_fdr = NA_character_,
                          top_gene_pct_used = pct_used))

    topN <- head(sig[order(sig$p.adjust), ], TOP_N_PATHWAYS)
    data.frame(top_pathways = paste(topN$ID, collapse = ORA_TERM_SEP),
               top_fdr = paste(formatC(topN$p.adjust, format = "e", digits = 2),
                               collapse = ORA_TERM_SEP),
               top_gene_pct_used = pct_used)
}

top_pathway_df <- dplyr::bind_rows(lapply(names(module_results), function(ds) {
    universe <- module_results[[ds]]$universe
    sel_ds   <- selected_all_df[selected_all_df$dataset == ds, ]
    references <- reference_sets_for(ds, sel_ds$cell_type)
    term2gene_full <- dplyr::bind_rows(lapply(names(references), function(term) {
        genes <- intersect(references[[term]], universe)
        if (length(genes) == 0) return(NULL)
        data.frame(term = term, gene = genes, stringsAsFactors = FALSE)
    }))

    dplyr::bind_rows(lapply(seq_len(nrow(sel_ds)), function(i) {
        row <- sel_ds[i, ]
        target_terms <- target_reference_terms(ds, row$cell_type)
        cbind(row[, c("dataset", "cell_type", "LV")],
              marker_source = "CellMarker + Allen + Azimuth",
              marker_reference_terms = paste(target_terms, collapse = " | "),
              run_lv_ora_topn(ds, row$LV, term2gene_full, universe))
    }))
}))

any_of_topn <- function(dataset, cell_type, top_str) {
    if (is.na(top_str)) return(FALSE)
    terms <- strsplit(top_str, ORA_TERM_SEP, fixed = TRUE)[[1]]
    any(terms %in% target_reference_terms(dataset, cell_type))
}
top_pathway_df$recovered <- mapply(
    any_of_topn, top_pathway_df$dataset, top_pathway_df$cell_type,
    top_pathway_df$top_pathways
)

top_pathway_table <- top_pathway_df
top_pathway_table$cell_type <- ct_label(top_pathway_table$cell_type)
top_pathway_table <- top_pathway_table[
    order(top_pathway_table$dataset, top_pathway_table$cell_type), ]
rownames(top_pathway_table) <- NULL

cat("Combined marker ORA recovered:", sum(top_pathway_table$recovered), "/",
    nrow(top_pathway_table), "\n")
top_pathway_table

In [ ]:
build_term2gene <- function(marker_sets) {
    dplyr::bind_rows(lapply(names(marker_sets), function(ct) {
        data.frame(term = ct, gene = marker_sets[[ct]], stringsAsFactors = FALSE)
    }))
}

run_ora_enrichment <- function(selected_lvs_df, Z, term2gene, universe, top_pct) {
    do.call(rbind, lapply(seq_len(nrow(selected_lvs_df)), function(i) {
        row       <- selected_lvs_df[i, ]
        top_genes <- get_top_genes(Z, row$LV, top_pct)
        res <- tryCatch(
            clusterProfiler::enricher(gene = top_genes, universe = universe,
                                      TERM2GENE = term2gene, pvalueCutoff = 1,
                                      qvalueCutoff = 1, minGSSize = 1, maxGSSize = Inf),
            error = function(e) NULL)
        if (is.null(res) || nrow(res@result) == 0) return(NULL)
        rdf <- res@result
        data.frame(lv_cell_type = row$cell_type, LV = row$LV,
                  marker_cell_type = rdf$ID, Count = rdf$Count,
                  pvalue = rdf$pvalue, FDR = rdf$p.adjust, stringsAsFactors = FALSE)
    }))
}

run_enrichment_for_datasets <- function(results_list, fun, ...) {
    dplyr::bind_rows(lapply(names(results_list), function(ds) {
        res       <- results_list[[ds]]
        z_path    <- file.path(MOD_ROOT, ds, "models", res$method, "Z.csv")
        Z         <- read.csv(z_path, row.names = 1, check.names = FALSE)
        term2gene <- build_term2gene(res$marker_sets)
        enr <- fun(res$selected, Z, term2gene, res$universe, ...)
        if (is.null(enr) || nrow(enr) == 0) return(NULL)
        enr$marker_reference_terms <- vapply(enr$marker_cell_type, function(ct)
            paste(target_reference_terms(ds, ct), collapse = " | "), character(1))
        cbind(dataset = ds, marker_source = res$marker_source, enr)
    }))
}

enrichment_res <- run_enrichment_for_datasets(module_results, run_ora_enrichment, TOP_GENE_PCT)

enrichment_res$neg_log10_fdr <- -log10(enrichment_res$FDR + 1e-300)

LABEL_MAX_TERMS <- 2
pick_label_hit <- function(dataset, cell_type, top_str, top_fdr_str) {
    if (is.na(top_str) || is.na(top_fdr_str))
        return(data.frame(top1_pathway = NA_character_, top1_fdr = NA_real_))
    paths <- strsplit(top_str, ORA_TERM_SEP, fixed = TRUE)[[1]]
    fdrs  <- as.numeric(strsplit(top_fdr_str, ORA_TERM_SEP, fixed = TRUE)[[1]])
    matches <- paths %in% target_reference_terms(dataset, cell_type)
    if (any(matches)) {
        ord   <- order(fdrs[matches])
        keep  <- head(ord, LABEL_MAX_TERMS)
        label <- paste(paths[matches][keep], collapse = " - ")
        fdr   <- min(fdrs[matches])
    } else {
        label <- paths[1]
        fdr   <- fdrs[1]
    }
    data.frame(top1_pathway = label, top1_fdr = fdr, stringsAsFactors = FALSE)
}
top_pathway_hits <- dplyr::bind_rows(mapply(
    pick_label_hit, top_pathway_df$dataset, top_pathway_df$cell_type,
    top_pathway_df$top_pathways, top_pathway_df$top_fdr,
    SIMPLIFY = FALSE
))
top_pathway_df$top1_pathway <- top_pathway_hits$top1_pathway
top_pathway_df$top1_fdr     <- top_pathway_hits$top1_fdr

shorten_marker_label <- function(x) {
    out <- sub("^(CellMarker|Azimuth|Allen)::", "", x)
    out <- sub("^(PBMC-L[123]|Motor Cortex-(class|subclass)|Heart-L[12]|Lung V2 \\(HLCA\\)-ann Level [12])-",
               "", out)
    out <- sub("microglia / Perivascular Macrophage", "Microglia", out,
               fixed = TRUE)
    out <- sub("^natural Killer Cell$", "Natural killer cell", out)
    out
}

# NA (nothing significant) shows only the LV code, never the true cell type.
top_pathway_df$top_pathway_label <- ifelse(
    is.na(top_pathway_df$top1_pathway), top_pathway_df$LV,
    shorten_marker_label(top_pathway_df$top1_pathway)
)

cat("Combined marker ORA enrichment tests (pooled sets):",
    nrow(enrichment_res), "\n")

In [ ]:
lv_effect_all <- lv_corr_full[, c("dataset", "cell_type", "LV", "cor")]
names(lv_effect_all) <- c("dataset", "marker_cell_type", "LV", "row_effect")

enr_all <- dplyr::bind_rows(lapply(names(module_results), function(ds) {
    res <- module_results[[ds]]
    enr <- enrichment_res[enrichment_res$dataset == ds, ]
    if (nrow(res$selected) == 0 || nrow(enr) == 0) return(NULL)

    eff <- lv_effect_all[lv_effect_all$dataset == ds,
                         c("dataset", "marker_cell_type", "LV", "row_effect")]
    merge(enr, eff, by = c("dataset", "marker_cell_type", "LV"), all.x = TRUE)
}))

Z_LIM <- max(abs(lv_effect_all$row_effect), na.rm = TRUE)
recovered_top_q <- -log10(top_pathway_df$top1_fdr[top_pathway_df$recovered %in% TRUE] + 1e-300)
Q_LIM <- max(c(enr_all$neg_log10_fdr, recovered_top_q), na.rm = TRUE)

module_panels <- lapply(names(module_results), function(ds) {
    res <- module_results[[ds]]
    sel <- res$selected
    enr <- enr_all[enr_all$dataset == ds, ]
    if (nrow(sel) == 0 || nrow(enr) == 0) return(NULL)

    ct_order <- sort(sel$cell_type)

    tp_ds  <- top_pathway_df[top_pathway_df$dataset == ds, ]
    tp_lut <- setNames(tp_ds$top_pathway_label, tp_ds$cell_type)
    col_labels <- setNames(
        ifelse(tp_lut[sel$cell_type] == sel$LV, sel$LV, paste0(tp_lut[sel$cell_type], " - ", sel$LV)),
        sel$cell_type)
    enr <- enr[enr$marker_cell_type %in% ct_order, ]
    enr$lv_col_label     <- factor(col_labels[enr$lv_cell_type], levels = unname(col_labels[ct_order]))
    enr$marker_row_label <- factor(ct_label(enr$marker_cell_type), levels = ct_label(ct_order))

    row_label_to_ct <- setNames(ct_order, ct_label(ct_order))
    col_label_to_lv <- setNames(sel$LV, unname(col_labels[sel$cell_type]))

    enr <- tidyr::complete(enr, marker_row_label, lv_col_label,
                           fill = list(neg_log10_fdr = 0))
    enr$marker_cell_type <- row_label_to_ct[as.character(enr$marker_row_label)]
    enr$LV               <- col_label_to_lv[as.character(enr$lv_col_label)]
    enr$lv_cell_type      <- sel$cell_type[match(enr$LV, sel$LV)]
    enr$is_diagonal       <- enr$lv_cell_type == enr$marker_cell_type

    enr$row_effect <- NULL
    effect_lookup <- lv_effect_all[lv_effect_all$dataset == ds,
                                   c("marker_cell_type", "LV", "row_effect")]
    enr <- merge(enr, effect_lookup, by = c("marker_cell_type", "LV"), all.x = TRUE)
    enr$row_effect[is.na(enr$row_effect)] <- 0

    rec_lookup <- top_pathway_df[top_pathway_df$dataset == ds,
                                 c("cell_type", "LV", "recovered", "top1_fdr")]
    names(rec_lookup) <- c("lv_cell_type", "LV", "recovered", "top1_fdr")
    enr <- merge(enr, rec_lookup, by = c("lv_cell_type", "LV"), all.x = TRUE)
    enr$top1_neg_log10_fdr <- -log10(enr$top1_fdr + 1e-300)
    recover_idx <- enr$is_diagonal %in% TRUE & enr$recovered %in% TRUE &
                   !is.na(enr$top1_neg_log10_fdr)
    enr$neg_log10_fdr[recover_idx] <- pmax(enr$neg_log10_fdr[recover_idx],
                                           enr$top1_neg_log10_fdr[recover_idx])
    diag_recovered <- enr[enr$is_diagonal %in% TRUE & enr$recovered %in% TRUE, ]

    n_samples <- nrow(read.csv(truth_v0_path(ds), row.names = 1, check.names = FALSE))

    panel_plot <- ggplot(enr, aes(x = lv_col_label, y = marker_row_label)) +
        geom_point(aes(size = neg_log10_fdr, color = row_effect)) +
        { if (nrow(diag_recovered) > 0)
            geom_point(data = diag_recovered, aes(size = neg_log10_fdr),
                       shape = 1, color = "black", stroke = 1.3) } +
        scale_size_continuous(limits = c(0, Q_LIM), range = c(1, 9),
                              name = expression("Combined marker ORA " * -log[10] ~ "(FDR)")) +
        scale_color_gradientn(colors = c("#b2182b", "#f4a582", "#f7f7f7", "#a1d99b", "#007a33"),
                              values = c(0, 0.35, 0.5, 0.65, 1),
                              limits = c(-Z_LIM, Z_LIM),
                              name = "LV-cell-type\nPearson r") +
        scale_x_discrete(drop = FALSE) +
        scale_y_discrete(drop = FALSE) +
        theme_bw(base_size = 9) +
        theme(axis.text.x = element_text(angle = 45, hjust = 1),
              plot.title  = element_text(face = "bold", size = 10, hjust = 0.5)) +
        labs(x = NULL, y = NULL, title = paste0(ds, " (n = ", n_samples, " samples)"))

    panel_export <- data.frame(
        dataset          = ds,
        marker_source    = res$marker_source,
        marker_row_label = as.character(enr$marker_row_label),
        marker_row_rank  = as.integer(enr$marker_row_label),
        lv_col_label     = as.character(enr$lv_col_label),
        lv_col_rank      = as.integer(enr$lv_col_label),
        neg_log10_fdr    = enr$neg_log10_fdr,
        row_effect       = enr$row_effect,
        diag_recovered   = enr$is_diagonal %in% TRUE & enr$recovered %in% TRUE,
        n_samples        = n_samples,
        Z_LIM            = Z_LIM,
        Q_LIM            = Q_LIM,
        stringsAsFactors = FALSE
    )

    list(plot = panel_plot, data = panel_export)
})
module_panels <- Filter(Negate(is.null), module_panels)

module_panel_export <- dplyr::bind_rows(lapply(module_panels, `[[`, "data"))

options(repr.plot.width = 20, repr.plot.height = 12)
p_module <- wrap_plots(lapply(module_panels, `[[`, "plot"), ncol = 3, guides = "collect") &
    theme(legend.position = "right")
print(p_module)

In [ ]:
write.csv(ovr_all_df,        file.path(OUT_DIR, "module_ovr_specificity.csv"),        row.names = FALSE)
write.csv(selected_all_df,   file.path(OUT_DIR, "module_selected_lvs.csv"),           row.names = FALSE)
write.csv(top_pathway_table, file.path(OUT_DIR, "module_top_pathways.csv"),           row.names = FALSE)
write.csv(enr_all,           file.path(OUT_DIR, "marker_enrichment_long.csv"),         row.names = FALSE)
write.csv(lv_effect_all,     file.path(OUT_DIR, "lv_marker_effects.csv"),              row.names = FALSE)
write.csv(top_pathway_df,    file.path(OUT_DIR, "marker_pathway_recovery.csv"),        row.names = FALSE)
write.csv(module_panel_export, file.path(OUT_DIR, "module_panel_ready.csv"),        row.names = FALSE)

cat("Module-level marker recovery outputs saved to:", OUT_DIR, "\n")


## Save outputs

In [ ]:
write.csv(top_lvs_df, file.path(OUT_DIR, "top_lvs_per_celltype.csv"), row.names = FALSE)
write.csv(
  top_lvs_df %>% dplyr::select(dataset, cell_type, LV, cor,
                                r_next_best_ct, r_next_best_lv, margin_ct, margin_lv),
  file.path(OUT_DIR, "lv_specificity_margins.csv"), row.names = FALSE
)
write.csv(assignment_quality, file.path(OUT_DIR, "lv_assignment_quality.csv"), row.names = FALSE)
write.csv(assignment_quality_by_dataset, file.path(OUT_DIR, "lv_assignment_quality_by_dataset.csv"), row.names = FALSE)

cat("All outputs saved to:", OUT_DIR, "\n")